## Data Preprocessing

### Importing Libraries

In [9]:
import os
import cv2
import glob
import numpy as np
from tqdm import tqdm # This gives us a nice progress bar

print("Libraries loaded for preprocessing!")

Libraries loaded for preprocessing!


### Defining the cropping function

In [10]:
def crop_brain_contour(image):
    # Convert the image to grayscale if it isn't already
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
        
    # Apply a slight blur to remove noise, then threshold to make the brain solid white and background black
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)
    
    # Clean up the thresholded image 
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)

    # Find the contours of the brain
    contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # If no contours are found (e.g., empty image), return original
    if not contours:
        return image
        
    # Find the largest contour (which should be the brain)
    c = max(contours, key=cv2.contourArea)
    
    # Find the extreme points of the contour
    extLeft = tuple(c[c[:, :, 0].argmin()][0])
    extRight = tuple(c[c[:, :, 0].argmax()][0])
    extTop = tuple(c[c[:, :, 1].argmin()][0])
    extBot = tuple(c[c[:, :, 1].argmax()][0])
    
    # Crop the image using these extreme points
    # We add a small buffer (e.g., 5 pixels) so we don't cut off the edges
    buffer = 15
    new_image = image[
        max(0, extTop[1] - buffer) : min(image.shape[0], extBot[1] + buffer),
        max(0, extLeft[0] - buffer) : min(image.shape[1], extRight[0] + buffer)
    ]
    
    return new_image

### Setup Paths and Parameters

In [11]:
# Use 'r' for Windows paths
RAW_DIR = r'E:\fourth_sem\brain_tumor_detection\datasets\01_raw'
PROCESSED_DIR = r'E:\fourth_sem\brain_tumor_detection\datasets\03_processed'

# 224x224 is the standard for Transfer Learning models (ResNet, VGG16, MobileNet)
IMG_SIZE = (224, 224) 

# The splits we want to process
dataset_splits = ['Training', 'Testing']

### The preprocessing loop

In [12]:
# Loop through Training and Testing
for split in dataset_splits:
    raw_split_path = os.path.join(RAW_DIR, split)
    processed_split_path = os.path.join(PROCESSED_DIR, split)
    
    # Create the split folder in the processed directory if it doesn't exist
    os.makedirs(processed_split_path, exist_ok=True)
    
    # Get all the tumor classes (glioma, meningioma, notumor, pituitary)
    if not os.path.exists(raw_split_path):
        print(f"Skipping {split}: Folder not found.")
        continue
        
    classes = os.listdir(raw_split_path)
    
    for tumor_class in classes:
        raw_class_path = os.path.join(raw_split_path, tumor_class)
        processed_class_path = os.path.join(processed_split_path, tumor_class)
        
        # Create the class folder in the processed directory
        os.makedirs(processed_class_path, exist_ok=True)
        
        # Get all images in this specific folder
        if os.path.isdir(raw_class_path):
            images = glob.glob(os.path.join(raw_class_path, '*.*'))
            
            print(f"Processing {split} -> {tumor_class} ({len(images)} images)...")
            
            # Use tqdm to show a progress bar
            for img_path in tqdm(images):
                # 1. Read the image
                img = cv2.imread(img_path)
                if img is None:
                    continue
                
                # 2. Crop the dead space
                cropped_img = crop_brain_contour(img)
                
                # 3. Resize to standard dimensions
                resized_img = cv2.resize(cropped_img, IMG_SIZE, interpolation=cv2.INTER_CUBIC)
                
                # 4. Save to the new directory
                filename = os.path.basename(img_path)
                save_path = os.path.join(processed_class_path, filename)
                cv2.imwrite(save_path, resized_img)

print("\nData Preprocessing Complete! All images are saved in 03_processed.")

Processing Training -> glioma (1400 images)...


100%|██████████| 1400/1400 [00:24<00:00, 57.11it/s]


Processing Training -> meningioma (1400 images)...


100%|██████████| 1400/1400 [00:26<00:00, 53.42it/s]


Processing Training -> notumor (1400 images)...


...

Processing Testing -> notumor (400 images)...


100%|██████████| 400/400 [00:09<00:00, 44.44it/s]


Processing Testing -> pituitary (400 images)...


100%|██████████| 400/400 [00:10<00:00, 38.40it/s]


Data Preprocessing Complete! All images are saved in 03_processed.
